# Multi-LLM-as-Judge: ADRD Medical AI
## Complete SMU SuperPOD Experiment Notebook

**Tailored to:** `mkotha` on SMU SuperPOD | Account: `xnluo_ai_chatbot_cognitive_0002`

| Phase | Cell | What happens |
|---|---|---|
| 0 | — | SSH + SOCKS proxy + tmux + srun (local terminal) |
| 1 | 1 | Verify GPU node, venv, paths |
| 2 | 2 | Install deps + tabulate fix |
| 3 | 3 | Load .env + HF cache |
| 3 | 4 | Download all 4 judge models |
| 4 | 5 | Kill any old vLLM + launch 4 fresh servers |
| 4 | 6 | Tail logs (optional check) |
| 5 | 7 | Health check all 4 judges (20 min timeout) |
| 6 | 8 | Repo structure check |
| 6 | 9 | Unit tests |
| 7 | 10 | Exp 1 — Dataset analysis |
| 8 | 11 | Exp 2 — Agreement analysis |
| 9 | 12 | Exp 3 — Rubric sensitivity |
| 10 | 13 | Exp 4 — Box plots |
| 11 | 14 | Results summary + CSV |
| 12 | 15 | Display figures inline |
| 13 | 16 | Push to GitHub |
| 14 | 17 | Cleanup / kill vLLM |

---

## Phase 0 — Terminal Commands (run BEFORE opening this notebook)

```bash
# 1. On your LOCAL machine:
ssh -C -D 8080 mkotha@superpod.smu.edu

# 2. On SuperPOD login node:
tmux new -s cs7325
# (to reattach later: tmux attach -t cs7325)

# 3. Request GPU node (inside tmux):
srun -A xnluo_ai_chatbot_cognitive_0002 -N1 -G2 -c32 --mem=192G --time=4:00:00 --pty $SHELL

# 4. On the GPU compute node:
cd /lustre/smuexa01/client/users/mkotha/CS7325

# 5. Clone repo (only first time):
git clone https://YOUR_GITHUB_TOKEN@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git
cd Multi_LLM_as_Judge_Medical_AI

# 6. Create .env file (only first time):
cat > .env << 'EOF'
HF_TOKEN=hf_xxxxxxxxxxxx
GITHUB_TOKEN=ghp_xxxxxxxxxxxx
LOCAL_VLLM_JUDGE1_URL=http://localhost:8001
LOCAL_VLLM_JUDGE2_URL=http://localhost:8002
LOCAL_VLLM_JUDGE3_URL=http://localhost:8003
LOCAL_VLLM_JUDGE4_URL=http://localhost:8004
EOF

# 7. Activate venv and launch JupyterLab:
source /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/activate
cd /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
jupyter lab --ip=0.0.0.0 --no-browser
# Open the printed URL in your browser (SOCKS proxy port 8080)
```


---
## Phase 1 — Verify GPU Node & Environment

In [1]:
# CELL 1: Verify GPU, venv, paths
import subprocess, os, sys, importlib
from pathlib import Path
from datetime import datetime

def shell(cmd, check=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (result.stdout + result.stderr).strip()
    if out: print(out)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return result.returncode

LUSTRE_BASE = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
ROOT        = LUSTRE_BASE / 'Multi_LLM_as_Judge_Medical_AI'
VENV        = LUSTRE_BASE / '.venv'
HF_CACHE    = LUSTRE_BASE / 'hf_models'
LOG_DIR     = LUSTRE_BASE / 'vllm_logs'
PYTHON      = str(VENV / 'bin' / 'python')

HF_CACHE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repo root : {ROOT}')
print(f'venv      : {VENV}')
print(f'HF cache  : {HF_CACHE}')
print(f'Python    : {sys.executable}')
print(f'Started   : {datetime.now().isoformat()}')

print('\n=== GPU Status (expect 2x A100 80GB) ===')
shell('nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader')

print('\n=== SLURM allocation ===')
shell('echo "Node: $SLURMD_NODENAME  Job: $SLURM_JOB_ID  GPUs: $SLURM_GPUS  CPUs: $SLURM_CPUS_ON_NODE  Mem: $SLURM_MEM_PER_NODE"')

print('\n=== venv active? ===')
venv_ok = str(VENV) in sys.executable
print(f'  {"✅ YES" if venv_ok else "❌ NO — run: source " + str(VENV) + "/bin/activate"}')

print('\n=== Lustre disk ===')
shell(f'df -h {LUSTRE_BASE} | tail -1')

Repo root : /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
venv      : /lustre/smuexa01/client/users/mkotha/CS7325/.venv
HF cache  : /lustre/smuexa01/client/users/mkotha/CS7325/hf_models
Python    : /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/python3
Started   : 2026-05-18T18:54:46.281001

=== GPU Status (expect 2x A100 80GB) ===
0, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
1, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
2, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
3, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB

=== SLURM allocation ===
Node: bcm-dgxa100-0002  Job: 446857  GPUs: 4  CPUs: 64  Mem: 196608

=== venv active? ===
  ✅ YES

=== Lustre disk ===
100.127.24.16@o2ib1,100.127.25.16@o2ib1:100.127.24.17@o2ib1,100.127.25.17@o2ib1:100.127.24.18@o2ib1,100.127.25.18@o2ib1:100.127.24.19@o2ib1,100.127.25.19@o2ib1:/smuexa01  748T   37T  703T   5% /lustre/smuexa01/client


0

---
## Phase 2 — Install Dependencies

In [2]:
# CELL 2: Install all deps + auto-fix missing packages
REQUIREMENTS = ROOT / 'requirements.txt'
print(f'Installing from: {REQUIREMENTS}')
shell(f'{sys.executable} -m pip install -r {REQUIREMENTS} -q')

print('\nInstalling / verifying vLLM...')
shell(f'{sys.executable} -m pip install vllm -q')

# Auto-fix known missing packages
ALWAYS_INSTALL = ['tabulate', 'ipywidgets']
for pkg in ALWAYS_INSTALL:
    shell(f'{sys.executable} -m pip install {pkg} -q')

# Verify
REQUIRED = ['vllm', 'httpx', 'datasets', 'huggingface_hub', 'pandas',
            'matplotlib', 'seaborn', 'dotenv', 'pytest', 'tabulate']
print('\n=== Package check ===')
all_ok = True
for pkg in REQUIRED:
    try:
        m = importlib.import_module(pkg.replace('-', '_'))
        ver = getattr(m, '__version__', '?')
        print(f'  ✅ {pkg:<20} {ver}')
    except ImportError:
        print(f'  ❌ {pkg} — FAILED')
        all_ok = False

print('\n✅ All packages OK' if all_ok else '\n❌ Some packages missing — re-run this cell')

Installing from: /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI/requirements.txt

Installing / verifying vLLM...

=== Package check ===
INFO 05-18 18:55:08 [__init__.py:239] Automatically detected platform cuda.
  ✅ vllm                 0.8.5
  ✅ httpx                0.28.1
  ✅ datasets             4.8.5
  ✅ huggingface_hub      0.30.2
  ✅ pandas               2.3.3
  ✅ matplotlib           3.10.9
  ✅ seaborn              0.13.2
  ✅ dotenv               ?
  ✅ pytest               9.0.3
  ✅ tabulate             0.10.0

✅ All packages OK


--- 
## Phase 3 — Download Judge Models

Models cached at `/lustre/.../hf_models/` — never fills `/home`.

| Judge | Model | Max context | GPU | Notes |
|---|---|---|---|---|
| medgemma  | `google/medgemma-4b-it`    | 4096 | GPU 0 | Gated — HF token required |
| biomistral| `BioMistral/BioMistral-7B` | 4096 | GPU 1 | |
| meditron  | `epfl-llm/meditron-7b`     | 2048 | GPU 0 | |
| medalpaca | `medalpaca/medalpaca-7b`   | 2048 | GPU 1 | Replaces BioMedLM (vLLM 0.8.5 incompatible) |


In [3]:
# CELL 3: Load .env, set HF cache, and patch ALL experiment configs
#          to use medalpaca instead of biomedlm
import os, json
from pathlib import Path
from dotenv import load_dotenv

ROOT     = Path('/lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI')
HF_CACHE = Path('/lustre/smuexa01/client/users/mkotha/CS7325/hf_models')

load_dotenv(ROOT / '.env')
os.environ['HF_HOME']            = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE / 'datasets')

HF_TOKEN = os.environ.get('HF_TOKEN')
GH_TOKEN = os.environ.get('GITHUB_TOKEN')

print(f'HF_TOKEN:      {"✅ set" if HF_TOKEN else "❌ NOT SET — add to .env"}')
print(f'GITHUB_TOKEN:  {"✅ set" if GH_TOKEN else "⚠️  not set — needed for git push"}')
print(f'HF_CACHE:      {HF_CACHE}')
shell(f'du -sh {HF_CACHE} 2>/dev/null || echo "(empty)"')

# ── Patch all experiment configs: replace biomedlm with medalpaca ────────────
OLD_JUDGE = {"id": "biomedlm",  "model": "stanford-crfm/BioMedLM",   "url": "http://localhost:8004"}
NEW_JUDGE = {"id": "medalpaca", "model": "medalpaca/medalpaca-7b",    "url": "http://localhost:8004"}

cfg_dir = ROOT / 'config' / 'configs'
patched = []
for cfg_path in sorted(cfg_dir.glob('config_exp*.json')):
    with open(cfg_path) as f:
        cfg = json.load(f)
    changed = False
    for i, j in enumerate(cfg.get('judges', [])):
        if j.get('id') == 'biomedlm' or j.get('model') == 'stanford-crfm/BioMedLM':
            cfg['judges'][i] = NEW_JUDGE
            changed = True
    if changed:
        with open(cfg_path, 'w') as f:
            json.dump(cfg, f, indent=2)
        patched.append(cfg_path.name)

if patched:
    print(f'\n✅ Patched {len(patched)} config(s): {patched}')
else:
    print('\n✅ All configs already reference medalpaca — no changes needed')

# ── Delete stale results that have all-zero skipped rows ─────────────────────
results_dir = ROOT / 'results'
stale_cleared = []
for results_file in results_dir.glob('exp*_*.json'):
    try:
        with open(results_file) as f:
            data = json.load(f)
        # Check if all results are skipped (stale from failed run)
        if isinstance(data, list) and len(data) > 0:
            all_skipped = all(
                all(r.get('agreement_class') == 'skipped' for r in block.get('results', []))
                for block in data
                if 'results' in block
            )
            if all_skipped and len(data) > 0:
                results_file.unlink()
                stale_cleared.append(results_file.name)
    except Exception:
        pass

if stale_cleared:
    print(f'✅ Cleared {len(stale_cleared)} stale result file(s): {stale_cleared}')
    print('   These had all-skipped rows from the failed biomedlm run — will re-run cleanly.')
else:
    print('✅ No stale result files found')

print('\n✅ Cell 3 complete — configs patched, stale results cleared')


HF_TOKEN:      ✅ set
GITHUB_TOKEN:  ✅ set
HF_CACHE:      /lustre/smuexa01/client/users/mkotha/CS7325/hf_models
179G	/lustre/smuexa01/client/users/mkotha/CS7325/hf_models

✅ Patched 3 config(s): ['config_exp2_agreement.json', 'config_exp3_rubric_sensitivity.json', 'config_exp4_boxplots.json']
✅ Cleared 2 stale result file(s): ['exp2_agreement_results.json', 'exp3_sensitivity_results.json']
   These had all-skipped rows from the failed biomedlm run — will re-run cleanly.

✅ Cell 3 complete — configs patched, stale results cleared


In [4]:
# CELL 4: Download all judge models (skips already-cached)
# NOTE: BioMedLM (stanford-crfm/BioMedLM) is NOT downloaded —
#       it has a hard incompatibility with vLLM 0.8.5
#       (AssertionError: scale_attn_by_inverse_layer_idx not supported).
#       medalpaca/medalpaca-7b is used instead as Judge 4.
from huggingface_hub import snapshot_download
import time

JUDGE_MODELS = [
    {'id': 'medgemma',   'hf_id': 'google/medgemma-4b-it',       'needs_token': True},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B',    'needs_token': False},
    {'id': 'meditron',   'hf_id': 'epfl-llm/meditron-7b',        'needs_token': False},
    {'id': 'medalpaca',  'hf_id': 'medalpaca/medalpaca-7b',       'needs_token': False},
    # BioMedLM removed — incompatible with vLLM 0.8.5
    # {'id': 'biomedlm', 'hf_id': 'stanford-crfm/BioMedLM', ...}
]

for m in JUDGE_MODELS:
    marker_dir = HF_CACHE / f'models--{m["hf_id"].replace("/", "--")}'
    if marker_dir.exists() and any(marker_dir.rglob('config.json')):
        print(f'✅ [{m["id"]:12s}] already cached → {marker_dir}')
        continue
    if m['needs_token'] and not HF_TOKEN:
        print(f'❌ [{m["id"]:12s}] requires HF_TOKEN — set in .env')
        continue
    print(f'\n⏬  Downloading [{m["id"]}] {m["hf_id"]} ...')
    t0 = time.time()
    try:
        snapshot_download(
            repo_id=m['hf_id'],
            cache_dir=str(HF_CACHE),
            token=HF_TOKEN if m['needs_token'] else None,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', 'rust_model*'],
        )
        print(f'  ✅ Done in {(time.time()-t0)/60:.1f} min')
    except Exception as e:
        print(f'  ❌ Failed: {e}')

print('\n=== Total model cache ===')
shell(f'du -sh {HF_CACHE}')


✅ [medgemma    ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--google--medgemma-4b-it
✅ [biomistral  ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--BioMistral--BioMistral-7B
✅ [meditron    ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--epfl-llm--meditron-7b
✅ [medalpaca   ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--medalpaca--medalpaca-7b

=== Total model cache ===
179G	/lustre/smuexa01/client/users/mkotha/CS7325/hf_models


0

---
## Phase 4 — Launch vLLM Servers

**4x A100 80GB — 1 dedicated GPU per model, all launched in parallel.**

- `--enforce-eager` skips CUDA graph compilation → ~45s startup per model (was ~5min)
- All 4 start simultaneously → total ready time ~90s (was ~20min)
- `HF_HUB_OFFLINE=1` + local snapshot paths → no HF auth needed at runtime
- BioMedLM → medalpaca (BioMedLM incompatible with vLLM 0.8.5)

| Judge | Model | GPU | Port | Max Len |
|---|---|---|---|---|
| medgemma  | `google/medgemma-4b-it`    | GPU 0 | 8001 | 4096 |
| biomistral| `BioMistral/BioMistral-7B` | GPU 1 | 8002 | 4096 |
| meditron  | `epfl-llm/meditron-7b`     | GPU 2 | 8003 | 2048 |
| medalpaca | `medalpaca/medalpaca-7b`   | GPU 3 | 8004 | 2048 |


In [5]:
# CELL 5: Launch all 4 vLLM judges in parallel (fast startup)
import time, subprocess, os, json
from pathlib import Path

LUSTRE_BASE = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
HF_CACHE    = LUSTRE_BASE / 'hf_models'
LOG_DIR     = LUSTRE_BASE / 'vllm_logs'
PYTHON      = str(LUSTRE_BASE / '.venv' / 'bin' / 'python')
ROOT        = LUSTRE_BASE / 'Multi_LLM_as_Judge_Medical_AI'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── Step 1: Full cleanup ─────────────────────────────────────────────────────
print('=' * 60)
print('Step 1: Killing any existing vLLM processes...')
subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(3)
subprocess.run(
    'kill -9 $(fuser /dev/nvidia0 /dev/nvidia1 /dev/nvidia2 /dev/nvidia3 2>/dev/null) 2>/dev/null || true',
    shell=True
)
time.sleep(5)
print('GPU memory after cleanup:')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)

# ── Step 2: Resolve local snapshot paths ────────────────────────────────────
print('\nStep 2: Resolving local snapshot paths...')

def get_snapshot_path(hf_id):
    cache_name = 'models--' + hf_id.replace('/', '--')
    snap_dir = HF_CACHE / cache_name / 'snapshots'
    if not snap_dir.exists():
        raise FileNotFoundError(f"Not cached: {hf_id} — run Cell 4 first.")
    snapshots = list(snap_dir.iterdir())
    if not snapshots:
        raise FileNotFoundError(f"Empty snapshot dir for {hf_id}")
    return str(sorted(snapshots, key=lambda p: p.stat().st_mtime)[-1])

# 4x A100 80GB — 1 dedicated GPU per model
# --enforce-eager skips CUDA graph compilation → startup ~45s vs ~5min
JUDGE_CONFIGS = [
    {'id': 'medgemma',   'hf_id': 'google/medgemma-4b-it',    'port': 8001, 'gpu': '0', 'max_len': 4096, 'extra': '--trust-remote-code --enforce-eager'},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B', 'port': 8002, 'gpu': '1', 'max_len': 4096, 'extra': '--enforce-eager'},
    {'id': 'meditron',   'hf_id': 'epfl-llm/meditron-7b',     'port': 8003, 'gpu': '2', 'max_len': 2048, 'extra': '--enforce-eager'},
    {'id': 'medalpaca',  'hf_id': 'medalpaca/medalpaca-7b',    'port': 8004, 'gpu': '3', 'max_len': 2048, 'extra': '--enforce-eager'},
]

snapshot_paths = {}
all_cached = True
for j in JUDGE_CONFIGS:
    try:
        snap = get_snapshot_path(j['hf_id'])
        snapshot_paths[j['id']] = snap
        print(f"  ✅ [{j['id']:12s}] {snap.split('/')[-1][:16]}...")
    except FileNotFoundError as e:
        print(f"  ❌ [{j['id']:12s}] {e}")
        all_cached = False

if not all_cached:
    raise RuntimeError("Some models not cached — run Cell 4 first.")

# ── Step 3: Patch ALL experiment config JSONs on disk ───────────────────────
# vLLM serves models under the LOCAL SNAPSHOT PATH, not the HuggingFace model ID.
# The config JSONs must use the snapshot path as the "model" field,
# otherwise vLLM returns 404 and every judge call fails → all rows skipped.
print('\nStep 3: Patching experiment config JSONs with snapshot paths...')

CONFIG_DIR = ROOT / 'config' / 'configs'
JUDGE_UPDATE = [
    {'id': 'medgemma',   'url': 'http://localhost:8001'},
    {'id': 'biomistral', 'url': 'http://localhost:8002'},
    {'id': 'meditron',   'url': 'http://localhost:8003'},
    {'id': 'medalpaca',  'url': 'http://localhost:8004'},
]
# Map old IDs that may appear in configs
ID_ALIASES = {
    'biomedlm': 'medalpaca',   # BioMedLM replaced by medalpaca
}

for cfg_file in sorted(CONFIG_DIR.glob('config_exp*.json')):
    with open(cfg_file) as f:
        cfg = json.load(f)

    if 'judges' not in cfg:
        continue

    changed = False
    new_judges = []
    for jcfg in cfg['judges']:
        jid = jcfg['id']
        # Remap old IDs
        canonical_id = ID_ALIASES.get(jid, jid)
        snap = snapshot_paths.get(canonical_id)
        if snap is None:
            print(f"  ⚠️  [{cfg_file.name}] unknown judge id '{jid}' — keeping as-is")
            new_judges.append(jcfg)
            continue

        url_match = next((j for j in JUDGE_UPDATE if j['id'] == canonical_id), None)
        new_entry = {
            'id':    canonical_id,
            'model': snap,          # ← local path, matches what vLLM serves
            'url':   url_match['url'] if url_match else jcfg['url'],
        }
        if new_entry != {'id': jid, 'model': jcfg.get('model'), 'url': jcfg.get('url')}:
            changed = True
        new_judges.append(new_entry)

    cfg['judges'] = new_judges
    with open(cfg_file, 'w') as f:
        json.dump(cfg, f, indent=2)
    status = '✅ patched' if changed else '✓ already ok'
    print(f"  {status}: {cfg_file.name}")

# ── Step 4: Clear stale cached results so experiments re-run fresh ───────────
print('\nStep 4: Clearing stale experiment results...')
RESULTS_DIR = ROOT / 'results'
stale_files = [
    'exp2_agreement_results.json',
    'exp3_sensitivity_results.json',
    'exp4_boxplot_data.json',
]
for fname in stale_files:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        fpath.unlink()
        print(f"  🗑️  Deleted stale: {fname}")
    else:
        print(f"  ✓ Not present: {fname}")

# Also clear observed_class from benchmark CSV so it re-evaluates
benchmark_csv = ROOT / 'benchmark_dataset' / 'agreement_benchmark.csv'
if benchmark_csv.exists():
    import csv, io
    with open(benchmark_csv, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if rows and 'observed_class' in rows[0]:
        for row in rows:
            row['observed_class'] = ''
            row['verified'] = ''
        with open(benchmark_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print(f"  🗑️  Cleared observed_class from agreement_benchmark.csv")

# ── Step 5: Launch ALL 4 in parallel ────────────────────────────────────────
print('\nStep 5: Launching all 4 judges in PARALLEL...')
print('--enforce-eager: skips CUDA graph compile → ready in ~60-90s total\n')

procs = {}
for j in JUDGE_CONFIGS:
    log_file = str(LOG_DIR / f'vllm_{j["id"]}.log')
    subprocess.run(f'fuser -k {j["port"]}/tcp 2>/dev/null || true', shell=True)

    cmd = (
        f'CUDA_VISIBLE_DEVICES={j["gpu"]} '
        f'HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 '
        f'HF_HOME={HF_CACHE} '
        f'{PYTHON} -m vllm.entrypoints.openai.api_server '
        f'--model {snapshot_paths[j["id"]]} '
        f'--port {j["port"]} --host 0.0.0.0 '
        f'--gpu-memory-utilization 0.85 '
        f'--max-model-len {j["max_len"]} '
        f'--dtype bfloat16 '
        f'{j["extra"]} '
        f'> {log_file} 2>&1'
    )
    proc = subprocess.Popen(cmd, shell=True)
    procs[j['id']] = proc
    print(f'  🚀 [{j["id"]:12s}] PID={proc.pid}  GPU={j["gpu"]}  port={j["port"]}')

JUDGE_URLS = {j['id']: f'http://localhost:{j["port"]}' for j in JUDGE_CONFIGS}
print('\nAll 4 launched. Run Cell 7 to wait for health checks (~90s).')
print('\nConfig JSONs patched — experiments will use correct model paths.')


Step 1: Killing any existing vLLM processes...
GPU memory after cleanup:
0, 0 MiB, 81154 MiB
1, 0 MiB, 81154 MiB
2, 0 MiB, 81154 MiB
3, 0 MiB, 81154 MiB

Step 2: Resolving local snapshot paths...
  ✅ [medgemma    ] 290cda5eeccbee13...
  ✅ [biomistral  ] 9a11e1ffa817c211...
  ✅ [meditron    ] d7d0a5ed929384a6...
  ✅ [medalpaca   ] fbb41b75d5a46ba4...

Step 3: Patching experiment config JSONs with snapshot paths...
  ✅ patched: config_exp2_agreement.json
  ✅ patched: config_exp3_rubric_sensitivity.json
  ✅ patched: config_exp4_boxplots.json

Step 4: Clearing stale experiment results...
  ✓ Not present: exp2_agreement_results.json
  ✓ Not present: exp3_sensitivity_results.json
  ✓ Not present: exp4_boxplot_data.json
  🗑️  Cleared observed_class from agreement_benchmark.csv

Step 5: Launching all 4 judges in PARALLEL...
--enforce-eager: skips CUDA graph compile → ready in ~60-90s total

  🚀 [medgemma    ] PID=14991  GPU=0  port=8001
  🚀 [biomistral  ] PID=14995  GPU=1  port=8002
  🚀 [medit

In [6]:
# CELL 6 (optional): Check startup progress — run anytime while waiting
import subprocess
from pathlib import Path

LOG_DIR = Path('/lustre/smuexa01/client/users/mkotha/CS7325/vllm_logs')
JUDGES  = [
    {'id': 'medgemma',   'gpu': '0', 'port': 8001},
    {'id': 'biomistral', 'gpu': '1', 'port': 8002},
    {'id': 'meditron',   'gpu': '2', 'port': 8003},
    {'id': 'medalpaca',  'gpu': '3', 'port': 8004},
]

for j in JUDGES:
    log_file = LOG_DIR / f'vllm_{j["id"]}.log'
    print(f'\n--- [{j["id"]}] GPU={j["gpu"]} port={j["port"]} ---')
    subprocess.run(f'tail -4 {log_file} 2>/dev/null || echo "(no log yet)"', shell=True)

print('\n=== GPU memory ===')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)



--- [medgemma] GPU=0 port=8001 ---

--- [biomistral] GPU=1 port=8002 ---

--- [meditron] GPU=2 port=8003 ---

--- [medalpaca] GPU=3 port=8004 ---

=== GPU memory ===
0, 0 MiB, 81154 MiB
1, 0 MiB, 81154 MiB
2, 0 MiB, 81154 MiB
3, 0 MiB, 81154 MiB


CompletedProcess(args='nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', returncode=0)

---
## Phase 5 — Health Check (wait for all 4 judges)

In [7]:
# CELL 7: Wait for all 4 judges — polls every 10s, 10 min max
import httpx, time, subprocess
from pathlib import Path

LOG_DIR = Path('/lustre/smuexa01/client/users/mkotha/CS7325/vllm_logs')

if 'JUDGE_URLS' not in dir():
    JUDGE_URLS = {
        'medgemma':   'http://localhost:8001',
        'biomistral': 'http://localhost:8002',
        'meditron':   'http://localhost:8003',
        'medalpaca':  'http://localhost:8004',
    }

MAX_WAIT = 600
POLL     = 10
ready    = {k: False for k in JUDGE_URLS}
ready_time = {}
t_start  = time.time()

print('Waiting for all 4 vLLM judges (--enforce-eager = ~60-90s startup)...\n')

while not all(ready.values()):
    elapsed = time.time() - t_start
    if elapsed > MAX_WAIT:
        print(f'\n⏱️  Timeout after {elapsed/60:.1f} min. Error summary:')
        for name in [k for k, v in ready.items() if not v]:
            log = LOG_DIR / f'vllm_{name}.log'
            print(f'\n--- [{name}] errors ---')
            subprocess.run(
                f'grep -i "error\|Error\|assert\|OOM\|No available" {log} | tail -8',
                shell=True
            )
        break

    for name, url in JUDGE_URLS.items():
        if ready[name]:
            continue
        try:
            r = httpx.get(f'{url}/health', timeout=3)
            if r.status_code == 200:
                ready[name] = True
                ready_time[name] = time.time() - t_start
                print(f'  ✅ {name:12s} ready in {ready_time[name]:.0f}s')
        except Exception:
            pass

    if not all(ready.values()):
        pending = [k for k, v in ready.items() if not v]
        print(f'  ⏳ [{time.time()-t_start:5.0f}s] Pending: {pending}', end='\r')
        time.sleep(POLL)

print()
if all(ready.values()):
    total = time.time() - t_start
    print(f'\n✅ ALL 4 JUDGES READY in {total:.0f}s ({total/60:.1f} min)')
    print('\nGPU memory:')
    subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)
    print('\n→ Proceed to Cell 8')
else:
    failed = [k for k, v in ready.items() if not v]
    print(f'\n❌ Failed: {failed} — check logs above, re-run Cell 5')


Waiting for all 4 vLLM judges (--enforce-eager = ~60-90s startup)...

  ✅ meditron     ready in 41smma', 'biomistral', 'meditron', 'medalpaca']
  ✅ biomistral   ready in 51smma', 'biomistral', 'medalpaca']
  ✅ medgemma     ready in 61smma', 'medalpaca']
  ✅ medalpaca    ready in 101saca']


✅ ALL 4 JUDGES READY in 101s (1.7 min)

GPU memory:
0, 68201 MiB, 12953 MiB
1, 69119 MiB, 12035 MiB
2, 69101 MiB, 12053 MiB
3, 69101 MiB, 12053 MiB

→ Proceed to Cell 8


---
## Phase 6 — Pre-Flight Checks

In [8]:
# CELL 8: Pre-flight check — verify all 4 judges serve the right models
import httpx, json
from pathlib import Path

ROOT = Path('/lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI')

# Load config to know what we expect
cfg_path = ROOT / 'config' / 'configs' / 'config_exp2_agreement.json'
with open(cfg_path) as f:
    cfg = json.load(f)

print('Pre-flight: verifying judges match config...\n')
all_ok = True
for j in cfg['judges']:
    port = int(j['url'].split(':')[-1])
    try:
        # Health check
        h = httpx.get(f'{j["url"]}/health', timeout=5)
        health_ok = h.status_code == 200

        # Model check — what is vLLM actually serving?
        m = httpx.get(f'{j["url"]}/v1/models', timeout=5)
        served_model = m.json()['data'][0]['id'] if m.status_code == 200 else 'unknown'

        # The served model ID will be the local snapshot path (not the HF ID)
        # so we just check health + reachability
        status = '✅' if health_ok else '❌'
        if not health_ok:
            all_ok = False
        print(f'  {status} {j["id"]:12s} port={port}  serving: {served_model[-50:]}')
    except Exception as e:
        print(f'  ❌ {j["id"]:12s} port={port}  UNREACHABLE: {e}')
        all_ok = False

if not all_ok:
    raise RuntimeError('\n❌ Not all judges are up — re-run Cell 5 then Cell 7.')
print('\n✅ All 4 judges healthy — ready for experiments')


Pre-flight: verifying judges match config...

  ✅ medgemma     port=8001  serving: snapshots/290cda5eeccbee130f987c4ad74a59ae6f196408
  ✅ biomistral   port=8002  serving: snapshots/9a11e1ffa817c211cbb52ee1fb312dc6b61b40a5
  ✅ meditron     port=8003  serving: snapshots/d7d0a5ed929384a6b059ac74198cf1d71f44ba76
  ✅ medalpaca    port=8004  serving: snapshots/fbb41b75d5a46ba405d496db1083a6f1d3df72a2

✅ All 4 judges healthy — ready for experiments


In [9]:
# CELL 9: Run unit tests (non-fatal — experiment cells run regardless)
import subprocess
from pathlib import Path

ROOT = Path('/lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI')
result = subprocess.run(
    f'cd {ROOT} && python -m pytest tests/ -x -q --tb=short 2>&1 | tail -20',
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('⚠️  Some tests failed — experiments may still run. Check output above.')
else:
    print('✅ All tests passed')


============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-9.0.3, pluggy-1.6.0
rootdir: /lustre/smuexa01/client/users/mkotha/CS7325
configfile: pyproject.toml
plugins: anyio-4.13.0
collected 6 items

tests/test_core.py ......                                                [100%]

=============================== warnings summary ===============================
../.venv/lib/python3.10/site-packages/_pytest/config/__init__.py:1434
  /lustre/smuexa01/client/users/mkotha/CS7325/.venv/lib/python3.10/site-packages/_pytest/config/__init__.py:1434: PytestConfigWarning: Unknown config option: asyncio_mode
  
    self._warn_or_fail_if_strict(f"Unknown config option: {key}\n")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
========================= 6 passed, 1 warning in 0.10s =========================

✅ All tests passed


---
## Phase 7 — Experiment 1: ADRD-Bench Dataset Analysis

In [10]:
# CELL 10: Experiment 1
import importlib, experiments.exp1_dataset_analysis as exp1_mod
importlib.reload(exp1_mod)

print('Running Experiment 1: ADRD-Bench Dataset Analysis')
print('='*60)
exp1_mod.main()

from collections import Counter
import pandas as pd
q_path = ROOT / 'benchmark_dataset' / 'adrd_questions.json'
with open(q_path) as f:
    qdata = json.load(f)
counts = Counter(q['category'] for q in qdata['questions'])
df_cat = pd.DataFrame([{'Category': k, 'N': v} for k, v in sorted(counts.items())])
df_cat.loc[len(df_cat)] = ['TOTAL', df_cat['N'].sum()]
print('\n=== Dataset Breakdown ===')
print(df_cat.to_markdown(index=False))
print(f'\n✅ Exp1 complete — {qdata["metadata"]["total_questions"]} questions')

Running Experiment 1: ADRD-Bench Dataset Analysis
Experiment 1: Structured Multi-Domain Medical Benchmark
Sources: MedQuAD | MedDialog | Medical Meadow Health Advice
Origin: https://github.com/m22oct2000/Multi-LLMs-as-Judge
Data dir: /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI/benchmark_dataset/source_datasets
Target per domain: 20


Total questions from real data: 0
Final benchmark size: 25 questions

Domain breakdown:
  Cardiology     :   5 questions
  Pharmacology   :   5 questions
  Neurology      :   5 questions
  Pediatrics     :   5 questions
  Emergency      :   5 questions

Source breakdown:
  placeholder         :  25

BENCHMARK TABLE (for paper):
Domain          Question Summary                                           #
----------------------------------------------------------------------------
Cardiology      STEMI management, 65-year-old, inferior ST elevation       5
Pharmacology    Metformin contraindications and renal safety criteria    

---
## Phase 8 — Experiment 2: Per-Rubric Agreement Analysis

In [11]:
# CELL 11: Experiment 2 — Per-Rubric Agreement Analysis
import importlib, sys, json
from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path('/lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI')
sys.path.insert(0, str(ROOT))

# ── Guard: clear stale all-skipped results so we don't skip re-running ───────
exp2_path = ROOT / 'results' / 'exp2_agreement_results.json'
if exp2_path.exists():
    with open(exp2_path) as f:
        existing = json.load(f)
    all_skipped = all(
        all(r.get('agreement_class') == 'skipped' for r in block.get('results', []))
        for block in existing if 'results' in block
    )
    if all_skipped:
        exp2_path.unlink()
        print('⚠️  Cleared stale all-skipped exp2 results — will re-run from scratch')
    else:
        done = [b['rubric_id'] for b in existing]
        print(f'Resuming: {len(done)} rubric(s) already have live results: {done}')

# ── Reload and run ────────────────────────────────────────────────────────────
import experiments.exp2_agreement_analysis as exp2_mod
importlib.reload(exp2_mod)

print('\nRunning Experiment 2: Per-Rubric Agreement Analysis')
print('=' * 60)
exp2_mod.main()

# ── Display results ───────────────────────────────────────────────────────────
with open(exp2_path) as f:
    exp2_data = json.load(f)

rows = []
for block in exp2_data:
    results = block['results']
    live    = [r for r in results if r.get('agreement_class') != 'skipped']
    skipped = len(results) - len(live)
    total_q = len(live)
    classes = Counter(r['agreement_class'] for r in live)
    mean_pw = sum(r['mean_pairwise_agreement'] for r in live) / total_q if total_q else 0
    rows.append({
        'Rubric':           block['rubric_name'].split('—')[0].strip()[:30],
        'N (live)':         total_q,
        'Skipped':          skipped,
        'Fully Agree %':    round(classes.get('fully_agree',    0) / total_q * 100, 1) if total_q else 0,
        'Majority Agree %': round(classes.get('majority_agree', 0) / total_q * 100, 1) if total_q else 0,
        'Split %':          round(classes.get('split',          0) / total_q * 100, 1) if total_q else 0,
        'Disagree %':       round(classes.get('full_disagree',  0) / total_q * 100, 1) if total_q else 0,
        'Mean PW%':         round(mean_pw, 1),
    })

df_exp2 = pd.DataFrame(rows)
print('\n=== Agreement Table ===')
print(df_exp2.to_markdown(index=False))

# ── Warn if still all zeros ───────────────────────────────────────────────────
if df_exp2['N (live)'].sum() == 0:
    print('\n❌ WARNING: All rows still skipped. Diagnosing...')
    import httpx
    for j in json.load(open(ROOT / 'config' / 'configs' / 'config_exp2_agreement.json'))['judges']:
        try:
            r = httpx.get(f'{j["url"]}/v1/models', timeout=5)
            model_id = r.json()['data'][0]['id']
            print(f'  port {j["url"].split(":")[-1]}: serving [{model_id[-40:]}]')
        except Exception as e:
            print(f'  port {j["url"].split(":")[-1]}: ERROR {e}')
    print('\nIf model IDs are local paths (snapshots/...) that is correct.')
    print('If any port is unreachable → re-run Cell 5 and Cell 7 first.')
else:
    total_live = df_exp2['N (live)'].sum()
    total_skip = df_exp2['Skipped'].sum()
    print(f'\n✅ Exp2 complete — {total_live} live judgments, {total_skip} skipped')


2026-05-18 18:57:37,133 - adrd_judge_runner - INFO - ADRDJudgeRunner: 4 judges
2026-05-18 18:57:37,136 - adrd_judge_runner - INFO -   medgemma | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--google--medgemma-4b-it/snapshots/290cda5eeccbee130f987c4ad74a59ae6f196408 | http://localhost:8001
2026-05-18 18:57:37,137 - adrd_judge_runner - INFO -   biomistral | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--BioMistral--BioMistral-7B/snapshots/9a11e1ffa817c211cbb52ee1fb312dc6b61b40a5 | http://localhost:8002
2026-05-18 18:57:37,137 - adrd_judge_runner - INFO -   meditron | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--epfl-llm--meditron-7b/snapshots/d7d0a5ed929384a6b059ac74198cf1d71f44ba76 | http://localhost:8003
2026-05-18 18:57:37,137 - adrd_judge_runner - INFO -   medalpaca | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--medalpaca--medalpaca-7b/snapshots/fbb41b75d5a46ba405d496db1083a6f1d3df72a2 | http://localhost:8004
2026-05-18 


Running Experiment 2: Per-Rubric Agreement Analysis
Loaded 15 questions from agreement benchmark CSV

######################################################################
RUBRIC: PEMAT — Patient Education Materials Assessment Tool
SOURCE: Shoemaker SJ, Wolf MS, Brach C. Development of the Patient Education Materials Assessment Tool (PEMAT): A new measure of understandability and actionability for print and audiovisual patient information. Patient Education and Counseling. 2014;96(3):395-403. PMID: 24973195.
######################################################################

  Q [Cardiology] expected=fully_agree: What is the first-line treatment for a STEMI?...


2026-05-18 18:57:37,294 - httpx - INFO - HTTP Request: GET http://localhost:8004/v1/models "HTTP/1.1 200 OK"
2026-05-18 18:57:37,295 - adrd_judge_runner - INFO - Pre-flight: 4/4 judges live: ['medgemma', 'biomistral', 'meditron', 'medalpaca']
2026-05-18 18:57:38,203 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 200 OK"
2026-05-18 18:57:38,203 - adrd_judge_runner - INFO - Judge biomistral returned 199 chars in 907ms
2026-05-18 18:57:39,292 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 200 OK"
2026-05-18 18:57:39,293 - adrd_judge_runner - INFO - Judge meditron returned 274 chars in 1996ms
2026-05-18 18:57:39,869 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 200 OK"
2026-05-18 18:57:39,870 - adrd_judge_runner - INFO - Judge medalpaca returned 725 chars in 2568ms
2026-05-18 18:57:44,119 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"


JUDGE: medgemma | RUBRIC: PEMAT — Patient Education Materials Assessment Tool | Q: FU_00000
Aggregate Score: 80.00
Raw (224 chars): U1: 1 | Uses plain language, defines medical terms.
U2: 1 | Presents information in a logical order.
U3: 1 | Focuses on key messages.
A1: 1 | Provides specific, actionable steps.
A2: 0 | Does not address barriers to action.

  [U1] score=1 | Uses plain language, defines medical terms.
  [U2] score=1 | Presents information in a logical order.
  [U3] score=1 | Focuses on key messages.
  [A1] score=1 | Provides specific, actionable steps.
  [A2] score=0 | Does not address barriers to action.

JUDGE: biomistral | RUBRIC: PEMAT — Patient Education Materials Assessment Tool | Q: FU_00000
Aggregate Score: 100.00
Raw (199 chars): U1: 1 | The answer is brief and to the point. It uses the words “primary PCI within 90 minutes of first medical contact” and “call cath lab immediately” to make it clear what the caregiver should do.
  [U1] score=1 | The answer is brief 

2026-05-18 18:57:45,083 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 200 OK"
2026-05-18 18:57:45,085 - adrd_judge_runner - INFO - Judge biomistral returned 169 chars in 791ms
2026-05-18 18:57:46,110 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 200 OK"
2026-05-18 18:57:46,111 - adrd_judge_runner - INFO - Judge meditron returned 314 chars in 1816ms
2026-05-18 18:57:46,681 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 200 OK"
2026-05-18 18:57:46,681 - adrd_judge_runner - INFO - Judge medalpaca returned 643 chars in 2385ms
2026-05-18 18:57:49,677 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-18 18:57:49,678 - adrd_judge_runner - INFO - Judge medgemma returned 321 chars in 5385ms
2026-05-18 18:57:49,679 - adrd_judge_metrics - INFO - [PEMAT — Patient Education Materials Assessment Tool] Q=FU_00001 J=medgemma: score=100.00 l

...


JUDGE: medgemma | RUBRIC: PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) | Q: FU_00012
Aggregate Score: 4.40
Raw (264 chars): U1: 5 | Uses everyday language, defines medical terms.
U2: 4 | Logical order, presents key steps.
U3: 5 | Focuses on key messages, avoids tangents.
A1: 5 | Provides specific, actionable steps.
A2: 3 | Addresses barriers to action, but could be more comprehensive.

  [U1] score=5 | Uses everyday language, defines medical terms.
  [U2] score=4 | Logical order, presents key steps.
  [U3] score=5 | Focuses on key messages, avoids tangents.
  [A1] score=5 | Provides specific, actionable steps.
  [A2] score=3 | Addresses barriers to action, but could be more comprehensive.

JUDGE: biomistral | RUBRIC: PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) | Q: FU_00012
Aggregate Score: 4.00
Raw (136 chars): U1: 4 | The answer uses words caregivers use in everyday conversation. “Epinephrine shot” is a common te

2026-05-18 19:03:08,728 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:08,729 - adrd_judge_runner - INFO - Judge biomistral returned 33 chars in 300ms
2026-05-18 19:03:08,901 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:08,902 - adrd_judge_runner - INFO - Judge medalpaca returned 39 chars in 472ms
2026-05-18 19:03:10,232 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:10,233 - adrd_judge_runner - INFO - Judge meditron returned 191 chars in 1803ms
2026-05-18 19:03:14,283 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:14,284 - adrd_judge_runner - INFO - Judge medgemma returned 451 chars in 5856ms
2026-05-18 19:03:14,285 - adrd_judge_metrics - INFO - [PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant)] Q=FU_00013 J


JUDGE: medgemma | RUBRIC: PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) | Q: FU_00013
Aggregate Score: 4.40
Raw (451 chars): U1: 5 | Uses common terms like "blood cultures" and "antibiotics" and defines them in plain language.
U2: 4 | Presents the steps in a logical order (blood cultures, antibiotics, fluid bolus, lactate).
U3: 5 | Focuses on the key steps in managing suspected sepsis.
A1: 5 | Provides specific, actionabl...
  [U1] score=5 | Uses common terms like "blood cultures" and "antibiotics" and defines them in plain language.
  [U2] score=4 | Presents the steps in a logical order (blood cultures, antibiotics, fluid bolus, lactate).
  [U3] score=5 | Focuses on the key steps in managing suspected sepsis.
  [A1] score=5 | Provides specific, actionable steps (blood cultures, antibiotics, fluid bolus, lactate).
  [A2] score=3 | Mentions hypotension, which is a barrier, but doesn't offer specific workarounds.

JUDGE: biomistral | RUBRIC: PEMAT-Liker

2026-05-18 19:03:14,989 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:14,990 - adrd_judge_runner - INFO - Judge medalpaca returned 39 chars in 539ms
2026-05-18 19:03:15,486 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:15,487 - adrd_judge_runner - INFO - Judge biomistral returned 267 chars in 1036ms
2026-05-18 19:03:16,261 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:16,262 - adrd_judge_runner - INFO - Judge meditron returned 372 chars in 1811ms
2026-05-18 19:03:17,944 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:17,945 - adrd_judge_runner - INFO - Judge medgemma returned 240 chars in 3495ms
2026-05-18 19:03:17,946 - adrd_judge_metrics - INFO - [PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant)] Q=FU_00014


JUDGE: medgemma | RUBRIC: PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) | Q: FU_00014
Aggregate Score: 4.80
Raw (240 chars): U1: 5 | Uses everyday language, defines medical terms.
U2: 5 | Logical order, key information first.
U3: 5 | Focuses on key messages.
A1: 5 | Provides specific, actionable steps.
A2: 4 | Addresses barriers to action (needle decompression).

  [U1] score=5 | Uses everyday language, defines medical terms.
  [U2] score=5 | Logical order, key information first.
  [U3] score=5 | Focuses on key messages.
  [A1] score=5 | Provides specific, actionable steps.
  [A2] score=4 | Addresses barriers to action (needle decompression).

JUDGE: biomistral | RUBRIC: PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) | Q: FU_00014
Aggregate Score: 4.00
Raw (267 chars): U1: 4 | The answer uses words caregivers use in everyday conversation. “Tracheal deviation” is a term that is familiar to laypeople. “Hypotension” is no

---
## Phase 9 — Experiment 3: Rubric Sensitivity

In [12]:
# CELL 12: Experiment 3 — Rubric Sensitivity
import importlib, sys, json, pandas as pd
from pathlib import Path
from collections import Counter

ROOT = Path('/lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI')
sys.path.insert(0, str(ROOT))

exp2_path = ROOT / 'results' / 'exp2_agreement_results.json'
if not exp2_path.exists():
    raise RuntimeError('Exp2 results not found — run Cell 11 first.')

import experiments.exp3_rubric_sensitivity as exp3_mod
importlib.reload(exp3_mod)

print('Running Experiment 3: Rubric Sensitivity')
print('=' * 60)
exp3_mod.main()

exp3_path = ROOT / 'results' / 'exp3_sensitivity_results.json'
with open(exp3_path) as f:
    exp3_data = json.load(f)

sens_rows = []
for block in exp3_data:
    for v in block['variants']:
        sens_rows.append({
            'Rubric':  block['rubric_name'].split('—')[0].strip()[:25],
            'Variant': v['scoring_variant'],
            'Mean%':   v['mean_pairwise_agreement'],
            'Std%':    v['std_pairwise_agreement'],
        })
print(pd.DataFrame(sens_rows).to_markdown(index=False))
print(f'\n✅ Exp3 complete')


2026-05-18 19:03:17,997 - adrd_judge_runner - INFO - ADRDJudgeRunner: 4 judges
2026-05-18 19:03:17,997 - adrd_judge_runner - INFO -   medgemma | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--google--medgemma-4b-it/snapshots/290cda5eeccbee130f987c4ad74a59ae6f196408 | http://localhost:8001
2026-05-18 19:03:17,997 - adrd_judge_runner - INFO -   biomistral | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--BioMistral--BioMistral-7B/snapshots/9a11e1ffa817c211cbb52ee1fb312dc6b61b40a5 | http://localhost:8002
2026-05-18 19:03:17,997 - adrd_judge_runner - INFO -   meditron | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--epfl-llm--meditron-7b/snapshots/d7d0a5ed929384a6b059ac74198cf1d71f44ba76 | http://localhost:8003
2026-05-18 19:03:17,998 - adrd_judge_runner - INFO -   medalpaca | /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--medalpaca--medalpaca-7b/snapshots/fbb41b75d5a46ba405d496db1083a6f1d3df72a2 | http://localhost:8004
2026-05-18 

Running Experiment 3: Rubric Sensitivity

Rubric: PEMAT — Patient Education Materials Assessment Tool | Variant: BINARY


2026-05-18 19:03:18,156 - httpx - INFO - HTTP Request: GET http://localhost:8004/v1/models "HTTP/1.1 200 OK"
2026-05-18 19:03:18,157 - adrd_judge_runner - INFO - Pre-flight: 4/4 judges live: ['medgemma', 'biomistral', 'meditron', 'medalpaca']
2026-05-18 19:03:18,512 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:18,514 - adrd_judge_runner - INFO - Judge medalpaca returned 58 chars in 355ms
2026-05-18 19:03:18,700 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:18,701 - adrd_judge_runner - INFO - Judge biomistral returned 138 chars in 543ms
2026-05-18 19:03:19,940 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:19,941 - adrd_judge_runner - INFO - Judge meditron returned 220 chars in 1782ms
2026-05-18 19:03:21,576 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
2


JUDGE: medgemma | RUBRIC: PEMAT — Patient Education Materials Assessment Tool [BINARY] | Q: q_0000
Aggregate Score: 80.00
Raw (224 chars): U1: 1 | Uses plain language, defines medical terms.
U2: 1 | Presents information in a logical order.
U3: 1 | Focuses on key messages.
A1: 1 | Provides specific, actionable steps.
A2: 0 | Does not address barriers to action.

  [U1] score=1 | Uses plain language, defines medical terms.
  [U2] score=1 | Presents information in a logical order.
  [U3] score=1 | Focuses on key messages.
  [A1] score=1 | Provides specific, actionable steps.
  [A2] score=0 | Does not address barriers to action.

JUDGE: biomistral | RUBRIC: PEMAT — Patient Education Materials Assessment Tool [BINARY] | Q: q_0000
Aggregate Score: 100.00
Raw (138 chars): U1: 1 | The answer uses words caregivers use in everyday conversation. “Activate STEMI protocol” is a phrase that is familiar to ED staff.
  [U1] score=1 | The answer uses words caregivers use in everyday conversation. “Act

2026-05-18 19:03:22,445 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:22,446 - adrd_judge_runner - INFO - Judge medalpaca returned 113 chars in 705ms
2026-05-18 19:03:22,589 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:22,590 - adrd_judge_runner - INFO - Judge biomistral returned 202 chars in 850ms
2026-05-18 19:03:23,534 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:23,535 - adrd_judge_runner - INFO - Judge meditron returned 318 chars in 1795ms
2026-05-18 19:03:25,037 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-18 19:03:25,038 - adrd_judge_runner - INFO - Judge medgemma returned 209 chars in 3299ms
2026-05-18 19:03:25,039 - adrd_judge_metrics - INFO - [PEMAT — Patient Education Materials Assessment Tool [BINARY]] Q=q_0001 J=medgemma: score=60

...

  [SKIPPED] Q=q_0019 rubric=rubric5_pemat_likert_scaled_0_10 — only 0 judge(s) alive: []
  [SKIPPED] Q=q_0020 rubric=rubric5_pemat_likert_scaled_0_10 — only 0 judge(s) alive: []


2026-05-18 19:22:44,667 - adrd_judge_runner - WARNING - Judge biomistral unreachable: [Errno 111] Connection refused
2026-05-18 19:22:44,705 - adrd_judge_runner - WARNING - Judge meditron unreachable: [Errno 111] Connection refused
2026-05-18 19:22:44,743 - adrd_judge_runner - WARNING - Judge medalpaca unreachable: [Errno 111] Connection refused
2026-05-18 19:22:44,744 - adrd_judge_runner - INFO - Pre-flight: 0/4 judges live: []
2026-05-18 19:22:44,744 - adrd_judge_runner - ERROR - Only 0 judges live for Q=q_0021 rubric=rubric5_pemat_likert_scaled_0_10 — skipping row (need >= 2)
2026-05-18 19:22:44,782 - adrd_judge_runner - WARNING - Judge medgemma unreachable: [Errno 111] Connection refused
2026-05-18 19:22:44,820 - adrd_judge_runner - WARNING - Judge biomistral unreachable: [Errno 111] Connection refused
2026-05-18 19:22:44,860 - adrd_judge_runner - WARNING - Judge meditron unreachable: [Errno 111] Connection refused
2026-05-18 19:22:44,900 - adrd_judge_runner - WARNING - Judge medal

  [SKIPPED] Q=q_0021 rubric=rubric5_pemat_likert_scaled_0_10 — only 0 judge(s) alive: []
  [SKIPPED] Q=q_0022 rubric=rubric5_pemat_likert_scaled_0_10 — only 0 judge(s) alive: []


2026-05-18 19:22:44,977 - adrd_judge_runner - WARNING - Judge biomistral unreachable: [Errno 111] Connection refused
2026-05-18 19:22:45,015 - adrd_judge_runner - WARNING - Judge meditron unreachable: [Errno 111] Connection refused
2026-05-18 19:22:45,053 - adrd_judge_runner - WARNING - Judge medalpaca unreachable: [Errno 111] Connection refused
2026-05-18 19:22:45,054 - adrd_judge_runner - INFO - Pre-flight: 0/4 judges live: []
2026-05-18 19:22:45,054 - adrd_judge_runner - ERROR - Only 0 judges live for Q=q_0023 rubric=rubric5_pemat_likert_scaled_0_10 — skipping row (need >= 2)
2026-05-18 19:22:45,092 - adrd_judge_runner - WARNING - Judge medgemma unreachable: [Errno 111] Connection refused
2026-05-18 19:22:45,130 - adrd_judge_runner - WARNING - Judge biomistral unreachable: [Errno 111] Connection refused
2026-05-18 19:22:45,168 - adrd_judge_runner - WARNING - Judge meditron unreachable: [Errno 111] Connection refused
2026-05-18 19:22:45,207 - adrd_judge_runner - WARNING - Judge medal

  [SKIPPED] Q=q_0023 rubric=rubric5_pemat_likert_scaled_0_10 — only 0 judge(s) alive: []
  [SKIPPED] Q=q_0024 rubric=rubric5_pemat_likert_scaled_0_10 — only 0 judge(s) alive: []
  Mean PW: 0.00%  Std: 0.00%  n=0

ABLATION — Indistinguishable scoring variants (std < 5%):
  ⚠️  Prometheus — Fine-Grained LLM Evaluation Rubric [LIKERT_1_5] std=0.0% — consider removing
  ⚠️  Prometheus — Fine-Grained LLM Evaluation Rubric [SCALED_0_10] std=0.0% — consider removing
  ⚠️  PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) [BINARY] std=0.0% — consider removing
  ⚠️  PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) [LIKERT_1_5] std=0.0% — consider removing
  ⚠️  PEMAT-Likert — Patient Education Materials Assessment Tool (Likert 1-5 Variant) [SCALED_0_10] std=0.0% — consider removing

Results saved to /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI/results/exp3_sensitivity_results.json
| Rubric                 

---
## Phase 10 — Experiment 4: Box Plots

In [13]:
# CELL 13: Experiment 4 — Box Plots
import importlib, sys
from pathlib import Path

ROOT = Path('/lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI')
sys.path.insert(0, str(ROOT))

for dep in ['exp2_agreement_results.json', 'exp3_sensitivity_results.json']:
    if not (ROOT / 'results' / dep).exists():
        raise RuntimeError(f'{dep} not found — run earlier experiment cells first.')

import experiments.exp4_boxplot_agreement as exp4_mod
importlib.reload(exp4_mod)

print('Running Experiment 4: Box Plot Generation')
print('=' * 60)
exp4_mod.main()

pngs = sorted((ROOT / 'results' / 'figures').glob('*.png'))
print(f'\n✅ Exp4 complete: {len(pngs)} figures')
for p in pngs:
    print(f'  📊 {p.name}  ({p.stat().st_size//1024} KB)')


RuntimeError: exp2_agreement_results.json not found — run earlier experiment cells first.

---
## Phase 11 — Results Summary

In [ ]:
# CELL 14: Final results summary — copy to paper
import json, pandas as pd
from pathlib import Path
from collections import Counter

ROOT = Path('/lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI')

print('=== FINAL RESULTS — COPY TO PAPER ===\n')

# Exp 1
try:
    q_path = ROOT / 'benchmark_dataset' / 'adrd_questions.json'
    with open(q_path) as f:
        qdata = json.load(f)
    counts = Counter(q['category'] for q in qdata['questions'])
    print('[Exp 1] Dataset:')
    for cat, n in sorted(counts.items()):
        print(f'  {cat}: {n}')
    print(f'  TOTAL: {sum(counts.values())}')
except FileNotFoundError:
    print('[Exp 1] adrd_questions.json not found — run Exp1 cell first')

# Exp 2
print()
try:
    with open(ROOT / 'results' / 'exp2_agreement_results.json') as f:
        exp2_data = json.load(f)
    rows = []
    for block in exp2_data:
        results = block['results']
        live    = [r for r in results if r.get('agreement_class') != 'skipped']
        total_q = len(live)
        classes = Counter(r['agreement_class'] for r in live)
        mean_pw = sum(r['mean_pairwise_agreement'] for r in live) / total_q if total_q else 0
        rows.append({
            'Rubric':           block['rubric_name'].split('—')[0].strip()[:30],
            'N (live)':         total_q,
            'Fully Agree %':    round(classes.get('fully_agree',    0) / total_q * 100, 1) if total_q else 0,
            'Majority Agree %': round(classes.get('majority_agree', 0) / total_q * 100, 1) if total_q else 0,
            'Split %':          round(classes.get('split',          0) / total_q * 100, 1) if total_q else 0,
            'Disagree %':       round(classes.get('full_disagree',  0) / total_q * 100, 1) if total_q else 0,
            'Mean PW%':         round(mean_pw, 1),
        })
    df_exp2 = pd.DataFrame(rows)
    print('[Exp 2] Agreement Table:')
    print(df_exp2.to_markdown(index=False))
    df_exp2.to_csv(ROOT / 'results' / 'agreement_summary_table.csv', index=False)
    print('\n✅ agreement_summary_table.csv saved')
except FileNotFoundError:
    print('[Exp 2] Results not found — run Exp2 cell first')

# Exp 3
print()
try:
    with open(ROOT / 'results' / 'exp3_sensitivity_results.json') as f:
        exp3_data = json.load(f)
    print('[Exp 3] Best scoring variant per rubric:')
    for block in exp3_data:
        best = max(block['variants'], key=lambda v: v['mean_pairwise_agreement'])
        print(f'  {block["rubric_name"][:30]:30s} → {best["scoring_variant"]} ({best["mean_pairwise_agreement"]:.1f}%)')
except FileNotFoundError:
    print('[Exp 3] Results not found — run Exp3 cell first')

# Exp 4
print()
try:
    pngs = sorted((ROOT / 'results' / 'figures').glob('*.png'))
    print(f'[Exp 4] {len(pngs)} PNG figures in results/figures/')
except Exception:
    print('[Exp 4] Figures not found — run Exp4 cell first')


---
## Phase 12 — Display Figures Inline

In [ ]:
# CELL 15: Display all PNG figures
from IPython.display import Image, display, Markdown
for png in sorted((ROOT / 'results' / 'figures').glob('*.png')):
    display(Markdown(f'### {png.stem}'))
    display(Image(filename=str(png), width=950))

---
## Phase 13 — Push Results to GitHub

In [ ]:
# CELL 16: Commit and push results
if GH_TOKEN:
    shell(f'git -C {ROOT} remote set-url origin https://{GH_TOKEN}@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git')
else:
    print('⚠️  GITHUB_TOKEN not set — push will prompt for password')

COMMIT_MSG = f'HPC results (SuperPOD mkotha): all 4 experiments — {datetime.now().strftime("%Y-%m-%d %H:%M")}'

print('=== Files to commit ===')
shell(f'git -C {ROOT} status --short')

shell(f'git -C {ROOT} add results/')
shell(f'git -C {ROOT} add benchmark_dataset/')
shell(f'git -C {ROOT} commit -m "{COMMIT_MSG}"')

rc = shell(f'git -C {ROOT} push origin main')
if rc == 0:
    print('\n✅ Pushed! → https://github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI/tree/main/results')
else:
    print(f'\n⚠️  Push failed. Try: cd {ROOT} && git push origin main')

---
## Phase 14 — Cleanup

In [ ]:
# CELL 17: Kill vLLM servers and free all GPUs
import subprocess, time

print('=== GPU before cleanup ===')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)

if 'procs' in dir():
    for name, proc in procs.items():
        try:
            proc.terminate()
            print(f'  Terminated [{name}] PID={proc.pid}')
        except Exception:
            pass

subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(3)
subprocess.run(
    'kill -9 $(fuser /dev/nvidia0 /dev/nvidia1 /dev/nvidia2 /dev/nvidia3 2>/dev/null) 2>/dev/null || true',
    shell=True
)
time.sleep(4)
print('\n=== GPU after cleanup ===')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)
print('\n✅ Done. All 4 GPUs free. Exit srun with: exit')


---
## ✅ All Experiments Complete

| File | Use |
|---|---|
| `results/agreement_summary_table.csv` | Paper Table 2 |
| `results/figures/*.png` | Upload to Overleaf |
| `results/exp2_agreement_results.json` | Full rationales |
| `benchmark_dataset/adrd_questions.json` | Dataset reference |

**Acknowledgement for paper:**
> *"Computational resources were provided by SMU's O'Donnell Data Science and Research Computing Institute (SuperPOD cluster, 2x NVIDIA A100 80GB GPUs, allocation: xnluo_ai_chatbot_cognitive_0002)."*
